# Bài 13 · Kết nối tắt & ResNet: thêm một phép cộng để mạng sâu học được

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/13-resnet.ipynb)

Notebook đi kèm [Bài 13 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/resnet/) (reel và demo tương tác nằm ở trang bài học).
Bài 12 kết thúc bằng một nghịch lý: trên MNIST, mạng tích chập “trơn” 56 lớp có lỗi *huấn luyện* cao hơn mạng 20 lớp.
Ta tự viết bằng numpy kết nối tắt $h \leftarrow h + F(h)$, đặt nó vào đúng mạng sâu của Bài 09–10, rồi xem gradient,
tín hiệu và kết quả huấn luyện thay đổi thế nào khi chỉ thêm đúng một phép cộng.

**Bạn sẽ làm:**
1. Đọc lại nghịch lý của mạng trơn sâu và lý do nó không phải quá khớp.
2. Viết một khối phần dư, kiểm tra rằng đạo hàm của $x + F(x)$ có một phần đồng nhất.
3. Đo gradient từng lớp của mạng MLP 30 lớp lúc khởi tạo, có và không có kết nối tắt, trên đúng 64 ảnh của demo.
4. Huấn luyện thật hai mạng 30 lớp trong 1 epoch, rồi đọc kết quả quét đầy đủ (MLP tới 100 lớp, CNN tới 56 lớp).
5. Xóa từng khối của một ResNet đã học (thí nghiệm của Veit và cộng sự, 2016) và so với mạng trơn.

**Cần biết trước:** [Bài 09 · Gradient biến mất & khởi tạo](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/vanishing-gradient/),
[Bài 10 · BatchNorm, LayerNorm & RMSNorm](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/chuan-hoa/) và
[Bài 12 · Pooling & kiến trúc CNN kinh điển](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/kien-truc-cnn/).
**Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải MNIST (~11 MB) lần đầu.

Mọi con số trong notebook đều do chính các ô bên dưới tính ra (hoặc đọc từ tệp kết quả mà demo dùng), và trùng với
số trong demo.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi. Hàm `sig()` in các tỉ
số gradient đúng như demo hiển thị chúng (899 · 38,6 · 7,3·10⁸).

In [ ]:
import base64
import gzip
import hashlib
import itertools
import json
import math
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, ScalarFormatter

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai:
# có kết nối tắt = xanh, nét liền, chấm tròn; mạng trơn = cam, nét gạch, ô vuông (như demo)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
RES = {"color": BLUE, "ls": "-", "marker": "o"}
PLAIN = {"color": ORANGE, "ls": (0, (5, 3)), "marker": "s"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SUP = str.maketrans("0123456789-", "⁰¹²³⁴⁵⁶⁷⁸⁹⁻")


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38."""
    s = (
        f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    )  # không in “−0”
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


def vn_eta(e):
    """η có một chữ số có nghĩa: 0,3 · 0,03 · 0,01."""
    return vn(e, max(0, -math.floor(math.log10(e) + 1e-9)))


def sig(x):
    """Như demo: 0,451 · 5,6 · 38,6 · 899 · 8.720; từ 10.000 trở lên viết dạng lũy thừa: 7,3·10⁸."""
    a = abs(x)
    if a == 0:
        return "0"
    if 1e-3 <= a < 1e4:
        return vn(x, 4 if a < 0.01 else 3 if a < 1 else 1 if a < 100 else 0)
    e = math.floor(math.log10(a))
    return f"{vn(x / 10**e, 1)}·10{str(e).translate(SUP)}"


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 0,001 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def pow_tick(v, _pos=None):
    return "1" if abs(math.log10(v)) < 1e-9 else f"10{str(round(math.log10(v))).translate(SUP)}"


def vn_axes(fig):
    """Đổi nhãn mọi trục sang kiểu Việt Nam. Trục log: vạch 1–2–5 nếu ngắn, lũy thừa của 10 nếu dài."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() == "log":
                lo, hi = sorted(lim)
                long = hi / lo >= 1000
                step = max(1, math.ceil(math.log10(hi / lo) / 6)) if long else 1
                ticks = [m * 10.0**k for k in range(-15, 16, step) for m in ((1,) if long else (1, 2, 5))]
                axis.set_major_locator(FixedLocator([t for t in ticks if lo <= t <= hi]))
                axis.set_minor_formatter(NullFormatter())
                axis.set_major_formatter(FuncFormatter(pow_tick if long else vn_tick))
            elif isinstance(
                axis.get_major_formatter(), ScalarFormatter
            ):  # giữ nguyên nhãn chữ (set_xticks(..., labels))
                axis.set_major_formatter(FuncFormatter(vn_tick))


print(vn(13007, 0), vn(0.1, 1), pct(0.9608), sig(899.3), sig(38.56), sig(7.348e8), sig(0.4512))

## 2 · Dữ liệu

MNIST (LeCun và cộng sự, 1998): 60.000 ảnh huấn luyện và 10.000 ảnh kiểm tra, chữ số viết tay xám 28×28. Ô dưới tải
4 tệp gốc và kiểm tra mã MD5, nên ai chạy cũng có đúng cùng một dữ liệu. Giống Bài 09–10, ta chia:
- **huấn luyện:** 55.000 ảnh đầu của tệp huấn luyện;
- **kiểm định:** 5.000 ảnh cuối, để *chọn* tốc độ học $\eta$ và để báo độ chính xác của các lần chạy 1 epoch.

Ô cũng nạp ba tệp kết quả mà demo dùng (`fetch_json`, có kiểm tra MD5). Chúng là các lần quét mất từ vài phút đến
hơn nửa giờ: mạng MLP sâu 10–100 lớp (`mnist-resmlp.json`) và mạng tích chập 8–56 lớp, trơn (`mnist-depth.json`,
Bài 12) lẫn có kết nối tắt (`mnist-resnet.json`).

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # mã MD5 chuẩn của MNIST
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


def load(kind):
    """Định dạng IDX: 16 byte đầu của tệp ảnh (8 byte của tệp nhãn) là phần mô tả, còn lại là các byte 0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# Kết quả của một lần huấn luyện dài (cùng tệp với demo trên web). build.py thay COURSE_DATA_URL bằng thư mục data/
# của repo notebook công khai.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


fetch()
mlp_runs = fetch_json("mnist-resmlp.json", "b05cf3f5374b56df6f33f3529031e117")  # MLP sâu: trơn / có kết nối tắt
cnn_plain = fetch_json("mnist-depth.json", "8156284d0611a3445408647bbae5d813")  # CNN trơn của Bài 12
cnn_res = fetch_json("mnist-resnet.json", "c52fd453cf49dd42e7b94e6e6486fdb0")  # cùng CNN, thêm kết nối tắt

In [ ]:
train_img, train_lab = load("train")
X_all, y_all = train_img.reshape(-1, 784) / 255.0, train_lab.astype(np.int64)
N_TRAIN, BATCH = 55000, 64
Xtr, ytr = X_all[:N_TRAIN], y_all[:N_TRAIN]  # huấn luyện
Xv, yv = X_all[N_TRAIN:], y_all[N_TRAIN:]  # kiểm định
print(f"huấn luyện {vn(len(ytr), 0)} · kiểm định {vn(len(yv), 0)} ảnh")
assert (len(ytr), len(yv), BATCH) == (mlp_runs["nTrain"], mlp_runs["nVal"], mlp_runs["batch"])

Phần đo gradient của demo dùng **lô huấn luyện đầu tiên của seed 1**: 64 ảnh đầu trong thứ tự xáo trộn
`default_rng(1).permutation(55000)`. Ô dưới lấy lại đúng lô đó từ MNIST và so với 64 ảnh mà demo lưu.

In [ ]:
idx_b = np.random.default_rng(1).permutation(N_TRAIN)[:BATCH]
xb, yb = Xtr[idx_b], ytr[idx_b]
demo_imgs = np.frombuffer(base64.b64decode(mlp_runs["images"]), np.uint8).reshape(BATCH, 784)
assert np.array_equal(demo_imgs, train_img.reshape(-1, 784)[idx_b]) and yb.tolist() == mlp_runs["labels"]
print("đúng 64 ảnh của demo; nhãn:", " ".join(map(str, yb[:16])), "…")

fig, axes = plt.subplots(4, 16, figsize=(10, 2.9))
for ax, x, lab in zip(axes.flat, xb, yb):
    ax.imshow(x.reshape(28, 28), cmap="gray_r")
    ax.set_title(str(lab), fontsize=9, pad=2)
    ax.axis("off")
fig.suptitle("Lô 64 ảnh dùng để đo gradient lúc khởi tạo (lô đầu tiên của seed 1)", fontsize=11)
plt.tight_layout()
vn_axes(fig)
plt.show()

## 3 · Nghịch lý: thêm lớp, lỗi huấn luyện tăng

Bài 12 đã dựng “bức tường độ sâu” theo đúng công thức của He và cộng sự (2015) cho CIFAR-10, thu nhỏ cho MNIST: một
lớp tích chập 3×3 đầu, ba chặng tích chập 3×3 (8, 16, 32 kênh), mỗi lớp là Conv → BatchNorm → ReLU, rồi lấy trung bình
toàn ảnh và một lớp dense 10. Độ sâu 8, 20, 32, 44, 56 lớp; SGD, lô 64, 3 epoch, 3 seed; $\eta$ chọn cho từng độ sâu
theo tập kiểm định. Ô dưới đọc lại kết quả đó và chọn $\eta$ đúng như demo: $\eta$ có lỗi kiểm định trung vị thấp nhất.

Con số cần nhìn là **lỗi huấn luyện** (đo sau khi học, trên 10.000 ảnh huấn luyện đầu tiên), không phải lỗi kiểm tra.

In [ ]:
def best_cnn(grid):
    """Mỗi độ sâu: η có lỗi kiểm định trung vị thấp nhất (seed nổ tính là 100%), như demo."""
    out = {}
    for d in grid["depths"]:
        cfg = {r["eta"]: r["seeds"] for r in grid["runs"] if r["depth"] == d}
        eta = min(grid["etas"], key=lambda e: np.median([s.get("val", 1.0) for s in cfg[e]]))
        out[d] = {"eta": eta, **{k: [s[k] for s in cfg[eta] if k in s] for k in ("train", "val", "test", "params")}}
    return out


def seed_text(v, d=2):
    """Trung vị [min–max] của các seed."""
    return f"{pct(np.median(v), d)} [{pct(min(v), d)}–{pct(max(v), d)}]"


P_cnn = best_cnn(cnn_plain)
print(f"lỗi huấn luyện đo trên {vn(cnn_plain['nTrainEval'], 0)} ảnh huấn luyện, {cnn_plain['epochs']} epoch")
for d, r in P_cnn.items():
    print(
        f"CNN trơn {d:2d} lớp ({vn(r['params'][0], 0):>7} tham số), η = {vn_eta(r['eta']):<4}: lỗi huấn luyện "
        f"{seed_text(r['train'])}, lỗi kiểm tra {seed_text(r['test'])}"
    )
long_plain = {(q["depth"], q["eta"]): q for q in cnn_plain["long"]}
for d in (20, 56):
    q = min((q for (dd, _), q in long_plain.items() if dd == d), key=lambda q: q["val"])
    print(f"10 epoch, seed 0, {d} lớp (η = {vn_eta(q['eta'])}): lỗi huấn luyện {pct(q['train'])}")

Hình dưới vẽ lỗi huấn luyện theo độ sâu: mỗi ô vuông cam là một seed, đường gạch nối các trung vị.

In [ ]:
depths = cnn_plain["depths"]
fig, ax = plt.subplots(figsize=(7.5, 3.2))
for i, d in enumerate(depths):
    ax.scatter(
        [d] * len(P_cnn[d]["train"]), 100 * np.array(P_cnn[d]["train"]), color=ORANGE, marker="s", s=28, zorder=3
    )
ax.plot(depths, [100 * np.median(P_cnn[d]["train"]) for d in depths], color=ORANGE, ls=PLAIN["ls"], lw=1.6)
ax.set(xticks=depths, xlabel="số lớp", ylabel="lỗi huấn luyện (%)", ylim=(0, None))
ax.set_title("CNN trơn trên MNIST (Bài 12): sâu hơn, lỗi huấn luyện cao hơn")
ax.grid(axis="y", alpha=0.3)
vn_axes(fig)
plt.show()

In [ ]:
def best_mlp(arch, norm, depth):
    """η có độ chính xác kiểm định trung vị cao nhất (seed nổ tính là 0), như demo; trả về η và các seed."""
    rs = [r for r in mlp_runs["runs"] if (r["arch"], r["norm"], r["depth"]) == (arch, norm, depth)]
    best = max(rs, key=lambda r: np.median([s.get("val", 0.0) for s in r["seeds"]]))
    return best["eta"], best["seeds"]


def mlp_text(seeds):
    v, n = [s["val"] for s in seeds if "val" in s], [s["nanStep"] for s in seeds if "nanStep" in s]
    if not v:
        return f"nổ (NaN) ở bước {min(n)}–{max(n)}" if min(n) != max(n) else f"nổ (NaN) ở bước {n[0]}"
    return seed_text(v, 1) + (f" ({len(n)}/{len(seeds)} seed nổ)" if n else "")


d20, d56 = P_cnn[20]["train"], P_cnn[56]["train"]
print(f"CNN: 56 lớp có lỗi huấn luyện cao hơn 20 lớp ở cả 3 seed: {'có' if min(d56) > max(d20) else 'không'}")
for norm in ("none", "bn"):
    eta, seeds = best_mlp("plain", norm, 30)
    print(
        f"MLP trơn 30 lớp, {'BatchNorm' if norm == 'bn' else 'không chuẩn hóa'} (η = {vn_eta(eta)}): "
        f"đúng {mlp_text(seeds)} trên tập kiểm định sau 1 epoch"
    )

Mạng 56 lớp chứa mạng 20 lớp: chỉ cần các lớp thêm vào làm ánh xạ đồng nhất (đầu ra bằng đầu vào), nó sẽ tốt đúng
bằng mạng 20 lớp. Vậy mà sau khi học, lỗi *huấn luyện* của nó cao hơn, ở cả 3 seed. Đây không phải **quá khớp**
(Bài 08): quá khớp là lỗi huấn luyện thấp mà lỗi kiểm tra cao. Ở đây chính việc tối ưu thất bại. He và cộng sự (2015)
gọi đó là **vấn đề suy thoái** (degradation) và thấy nó cả trên ImageNet: mạng trơn 34 lớp có lỗi top‑1 28,54%, tệ
hơn mạng trơn 18 lớp (27,94%), và lỗi huấn luyện của nó cao hơn trong suốt quá trình học (Bảng 2 và Hình 4 của bài báo).

Ba lối giải thích quen thuộc đều không khớp:
- *“Học chưa đủ lâu.”* 10 epoch (ô trên) không xóa được khoảng cách; bài báo cũng thử học gấp 3 lần số bước.
- *“Gradient biến mất.”* Các mạng này đều có BatchNorm; He và cộng sự kiểm tra rằng tín hiệu xuôi lẫn gradient ngược
  không biến mất. Mục 5 sẽ cho thấy chuyện xảy ra lúc khởi tạo còn trái ngược hơn: gradient *bùng nổ*.
- *“Thêm BatchNorm là đủ.”* Bài 10 đã thấy với MLP 30 lớp (hai dòng cuối ô trên): có hay không có BatchNorm, mạng trơn
  đều còn xa mức 96% của mạng 10 lớp (mục 8). 66,0% so với 77,0% thì nằm trong dao động giữa các seed, nên không nói
  được BatchNorm làm nó tốt hơn hay tệ hơn; chỉ biết là nó không cứu được.

## 4 · Khối phần dư: $y = x + F(x)$

Ý tưởng của ResNet: đừng bắt vài lớp học thẳng ánh xạ mong muốn $H(x)$; cho chúng học **phần chênh lệch**
$F(x) = H(x) - x$, rồi cộng $x$ vào bằng một **kết nối tắt** (shortcut):

$$y = x + F(x)$$

Phép cộng không có tham số nào. Ta dùng dạng “tiền kích hoạt” của He và cộng sự (2016), đúng như mạng MLP của demo:
nhánh $F$ là chuẩn hóa → ReLU → lớp tuyến tính, $F(x) = \mathrm{ReLU}(N(x))\,W + b$, còn đường tắt là phép đồng nhất
thuần túy. Ở mục này bỏ $N$ cho gọn. So với một lớp “trơn” $y = \mathrm{ReLU}(xW + b)$ có cùng $W$, $b$:

### 🤔 Dự đoán trước
Đặt mọi trọng số và độ lệch bằng 0: $W = 0$, $b = 0$. Lớp trơn trả về gì? Khối phần dư trả về gì? Và gradient
$\partial L/\partial x$ đi qua mỗi khối còn lại bao nhiêu?

In [ ]:
def relu(a):
    return np.maximum(0, a)


def vec(a, d=2):
    return "[" + "; ".join(vn(v, d) for v in a) + "]"


x = np.random.default_rng(0).normal(size=(1, 5))  # một ví dụ, 5 đơn vị
W0, b0 = np.zeros((5, 5)), np.zeros(5)
dy = np.ones((1, 5))  # gradient ∂L/∂y từ phía sau gửi về

y_plain = relu(x @ W0 + b0)
y_res = x + relu(x) @ W0 + b0
dx_plain = (dy * (x @ W0 + b0 > 0)) @ W0.T  # lượt ngược qua ReLU(xW + b)
dx_res = dy + (dy @ W0.T) * (x > 0)  # lượt ngược qua x + ReLU(x)W + b: đường tắt + nhánh F
print("x                 =", vec(x[0]))
print("lớp trơn,    W = 0: y =", vec(y_plain[0]), "  ∂L/∂x =", vec(dx_plain[0]))
print("khối phần dư, W = 0: y =", vec(y_res[0]), "  ∂L/∂x =", vec(dx_res[0]))
assert np.array_equal(y_res, x) and np.array_equal(dx_res, dy) and not y_plain.any() and not dx_plain.any()

Với $W = 0$, lớp trơn xóa sạch tín hiệu: $y = \mathrm{ReLU}(0) = 0$, và gradient gửi về cũng bằng 0. Khối phần dư thì
trả về đúng $x$, và gradient đi qua nguyên vẹn. Nghĩa là **“không làm gì” là trạng thái dễ nhất** của một khối phần dư:
nếu một khối không giúp gì, quá trình học chỉ cần kéo $F$ về gần 0. Một chồng lớp trơn muốn làm ánh xạ đồng nhất thì
phải tự học được $\mathrm{ReLU}(xW + b) = x$, khó hơn nhiều.

Tổng quát hơn, theo quy ước vectơ hàng $y_j = x_j + \sum_i \mathrm{ReLU}(x_i)\,W_{ij} + b_j$, nên

$$\frac{\partial y_j}{\partial x_i} = \underbrace{\delta_{ij}}_{\text{đường tắt}} + \underbrace{\mathbb 1[x_i > 0]\,W_{ij}}_{\text{nhánh } F}
\qquad\Longrightarrow\qquad \frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} + \Big(\frac{\partial L}{\partial y}\,W^\top\Big) \odot \mathbb 1[x > 0]$$

Ma trận Jacobi là $I + \partial F/\partial x$: phần $I$ chuyển gradient về lớp trước *không qua phép nhân nào*.
Ô dưới kiểm tra công thức đó bằng sai phân hữu hạn, với $W$ ngẫu nhiên.

In [ ]:
rng = np.random.default_rng(1)
W1, b1 = rng.normal(0, 0.5, (5, 5)), rng.normal(0, 0.1, 5)


def block(x):
    return x + relu(x) @ W1 + b1


J_formula = np.eye(5) + (x[0] > 0)[:, None] * W1  # J[i, j] = ∂y_j/∂x_i
J_numeric = np.zeros((5, 5))
for i in range(5):
    e = np.zeros((1, 5))
    e[0, i] = 1e-6
    J_numeric[i] = (block(x + e) - block(x - e))[0] / 2e-6
print(f"sai khác lớn nhất giữa công thức và sai phân hữu hạn: {sig(np.abs(J_formula - J_numeric).max())}")
assert np.allclose(J_formula, J_numeric, atol=1e-8)
print("đường chéo của J (luôn có phần 1 từ đường tắt):", vec(np.diag(J_formula)))

## 5 · Gradient đi qua phép cộng: mạng MLP 30 lớp lúc khởi tạo

Giờ đặt khối đó vào một mạng thật: mạng MLP sâu của Bài 09–10, 784 → 30×64 → 10, khởi tạo He. Ta dựng hai mạng với
**đúng cùng trọng số**, chỉ khác nhau ở phép cộng:
- **trơn:** $h \leftarrow \mathrm{ReLU}(N(hW + b))$ ở mọi lớp;
- **có kết nối tắt:** $h \leftarrow h + \mathrm{ReLU}(N(h))\,W + b$ từ lớp 2 (khối tiền kích hoạt); lớp 1
  (784 → 64) vẫn là lớp trơn vì kích thước đổi, và $N$ → ReLU chạy thêm một lần trước lớp ra.

$N$ là BatchNorm (Bài 10) hoặc không có gì. Để mạng trùng từng con số với demo, trọng số được rút như demo rút trong
trình duyệt: bộ sinh số `mulberry32(1)` của JavaScript, rồi phép Box–Muller biến mỗi cặp số đều thành một số chuẩn.

In [ ]:
WIDTH, EPS, MOM = 64, 1e-5, 0.1  # như Bài 10: độ rộng, ε của BatchNorm, động lượng của trung bình trượt


def mulberry32(seed, n):
    """n số đều đầu tiên trong [0, 1) của mulberry32(seed), bộ sinh số mà demo dùng (viết lại bằng numpy)."""
    a = ((seed + np.arange(1, n + 1, dtype=np.uint64) * 0x6D2B79F5) & 0xFFFFFFFF).astype(np.uint32)
    t = (a ^ (a >> 15)) * (a | 1)
    t = (t + (t ^ (t >> 7)) * (t | 61)) ^ t
    return (t ^ (t >> 14)).astype(np.float64) / 2**32


def he_init(depth, seed=1):
    """Thứ tự của demo: ε_ra (64×10), ε₁ (784×64), ε₂, ε₃… (64×64), lấy liên tiếp từ một dòng số chuẩn.
    Mạng sâu hơn chỉ nối thêm lớp, nên 30 lớp đầu của mạng 100 lớp trùng với mạng 30 lớp."""
    shapes = [(WIDTH, 10), (784, WIDTH)] + [(WIDTH, WIDTH)] * (depth - 1)
    sizes = [r * c for r, c in shapes]
    u = mulberry32(seed, 2 * sum(sizes))
    z = np.sqrt(-2 * np.log(1 - u[0::2])) * np.cos(2 * np.pi * u[1::2])  # Box–Muller
    eps = [z[k - n : k].reshape(s) for s, n, k in zip(shapes, sizes, itertools.accumulate(sizes))]
    return [np.sqrt(2 / r) * e for (r, _), e in zip(shapes[1:], eps[1:])] + [np.sqrt(2 / WIDTH) * eps[0]]  # He


W_demo = he_init(30)
print("W₁:", W_demo[0].shape, " W₂…W₃₀:", W_demo[1].shape, "×", len(W_demo) - 2, " W_ra:", W_demo[-1].shape)
print(f"độ lệch chuẩn của W₁: {vn(W_demo[0].std(), 4)} (He: √(2/784) = {vn(np.sqrt(2 / 784), 4)})")

Hai mạng dùng chung một danh sách trọng số; BatchNorm bắt đầu với $\gamma = 1$, $\beta = 0$ và giữ trung bình trượt
(động lượng 0,1) để dùng khi suy luận, như Bài 10. Mạng có kết nối tắt cần $L + 1$ lớp chuẩn hóa: lớp 1, một lớp trước
nhánh của mỗi khối, và lớp cuối.

In [ ]:
def make_net(arch, norm, depth, seed=1):
    k = depth + (arch == "res")  # số lớp chuẩn hóa
    W = he_init(depth, seed)
    return {
        "arch": arch,
        "norm": norm,
        "depth": depth,
        "W": W,
        "b": [np.zeros(w.shape[1]) for w in W],
        "g": [np.ones(WIDTH) for _ in range(k)],
        "be": [np.zeros(WIDTH) for _ in range(k)],
        "rm": [np.zeros(WIDTH) for _ in range(k)],
        "rv": [np.ones(WIDTH) for _ in range(k)],
    }


def norm_fwd(P, l, a, mode):
    """Lớp chuẩn hóa thứ l. mode 'train': thống kê của lô + cập nhật trung bình trượt; 'eval': trung bình trượt."""
    if P["norm"] == "none":
        return a, None
    if mode == "eval":
        return P["g"][l] * ((a - P["rm"][l]) / np.sqrt(P["rv"][l] + EPS)) + P["be"][l], None
    mu, var = a.mean(0), a.var(0)
    inv = 1 / np.sqrt(var + EPS)
    if mode == "train":
        P["rm"][l] = (1 - MOM) * P["rm"][l] + MOM * mu
        P["rv"][l] = (1 - MOM) * P["rv"][l] + MOM * var * len(a) / (len(a) - 1)
    n = (a - mu) * inv
    return P["g"][l] * n + P["be"][l], (n, inv)


def norm_bwd(P, l, dy, c, grads):
    """Lượt ngược qua BatchNorm (Bài 10); không chuẩn hóa thì gradient đi qua nguyên vẹn."""
    if c is None:
        return dy
    n, inv = c
    grads["g"][l], grads["be"][l] = (dy * n).sum(0), dy.sum(0)
    dn = dy * P["g"][l]
    return inv / len(dn) * (len(dn) * dn - dn.sum(0) - n * (dn * n).sum(0))

In [ ]:
def forward(P, x, mode="train"):
    """Trả về cache cho lượt ngược, h cuối, logit z và rms(h) sau từng lớp ẩn."""
    W, b, L = P["W"], P["b"], P["depth"]
    y, c = norm_fwd(P, 0, x @ W[0] + b[0], mode)  # lớp 1 (784 → 64): lớp trơn ở cả hai mạng
    cache, h = [(x, y, c)], relu(y)
    rms = [np.sqrt((h**2).mean())]
    for l in range(1, L):
        if P["arch"] == "plain":  # h ← ReLU(N(hW + b))
            y, c = norm_fwd(P, l, h @ W[l] + b[l], mode)
            cache.append((h, y, c))
            h = relu(y)
        else:  # h ← h + ReLU(N(h))·W + b
            y, c = norm_fwd(P, l, h, mode)
            r = relu(y)
            cache.append((r, y, c))
            h = h + r @ W[l] + b[l]
        rms.append(np.sqrt((h**2).mean()))
    if P["arch"] == "res":  # N → ReLU một lần nữa trước lớp ra
        y, c = norm_fwd(P, L, h, mode)
        cache.append((None, y, c))
        h = relu(y)
    return cache, h, h @ W[-1] + b[-1], rms


def softmax(z):
    e = np.exp(z - z.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

In [ ]:
def backward(P, cache, h, z, y):
    """Lan truyền ngược (Bài 06) cho cả hai mạng; trả về gradient của mọi tham số."""
    W, L = P["W"], P["depth"]
    dz = softmax(z)
    dz[np.arange(len(y)), y] -= 1
    dz /= len(y)  # ∂L/∂z của softmax + entropy chéo, trung bình trên lô
    k = len(P["g"])
    g = {"W": [None] * (L + 1), "b": [None] * (L + 1), "g": [None] * k, "be": [None] * k}
    g["W"][-1], g["b"][-1] = h.T @ dz, dz.sum(0)
    dh = dz @ W[-1].T
    if P["arch"] == "res":
        _, yl, c = cache[L]
        dh = norm_bwd(P, L, dh * (yl > 0), c, g)
    for l in range(L - 1, 0, -1):
        inp, yl, c = cache[l]
        if P["arch"] == "plain":  # gradient phải đi qua lớp: nhân với Wᵀ, BN và ReLU
            da = norm_bwd(P, l, dh * (yl > 0), c, g)
            g["W"][l], g["b"][l] = inp.T @ da, da.sum(0)
            dh = da @ W[l].T
        else:  # đường tắt chuyển dh về nguyên vẹn; nhánh F cộng thêm phần của nó
            g["W"][l], g["b"][l] = inp.T @ dh, dh.sum(0)
            dh = dh + norm_bwd(P, l, (dh @ W[l].T) * (yl > 0), c, g)
    x, yl, c = cache[0]
    da = norm_bwd(P, 0, dh * (yl > 0), c, g)
    g["W"][0], g["b"][0] = x.T @ da, da.sum(0)
    return g


def grad_profile(arch, norm, depth):
    """‖∂L/∂W_l‖ của mọi lớp ẩn và rms(h) lúc khởi tạo, trên lô 64 ảnh của demo (BN dùng thống kê của lô)."""
    P = make_net(arch, norm, depth)
    cache, h, z, rms = forward(P, xb, "batch")
    gW = backward(P, cache, h, z, yb)["W"][:depth]
    return np.array([np.linalg.norm(gw) for gw in gW]), np.array(rms)

### 🤔 Dự đoán trước
Mạng trơn 30 lớp có BatchNorm, lúc khởi tạo. Gradient của trọng số ở **lớp 1** lớn hơn hay nhỏ hơn ở **lớp 30**, và
khoảng bao nhiêu lần? Thêm kết nối tắt (cùng trọng số) thì tỉ số đó thành bao nhiêu?

In [ ]:
prof = {(a, n): grad_profile(a, n, 30) for a in ("plain", "res") for n in ("bn", "none")}
for a, name in (("plain", "trơn          "), ("res", "có kết nối tắt")):
    gw, rms = prof[(a, "bn")]
    print(
        f"BatchNorm, {name}: ‖∂L/∂W₁‖ / ‖∂L/∂W₃₀‖ = {sig(gw[0] / gw[-1]):>4}   "
        f"rms(h): lớp 1 {sig(rms[0])} → lớp 30 {sig(rms[-1])}"
    )
ratio_plain = prof[("plain", "bn")][0][0] / prof[("plain", "bn")][0][-1]
ratio_res = prof[("res", "bn")][0][0] / prof[("res", "bn")][0][-1]
print(f"mạng trơn có BatchNorm: mỗi lần lùi qua một lớp, gradient to thêm trung bình ×{vn(ratio_plain ** (1 / 29), 3)}")
# resmlp.py tính đúng hai tỉ số này trên cùng lô (899,343 và 38,5606); demo hiển thị chúng là 899 và 38,6
assert abs(ratio_plain / 899.343 - 1) < 1e-5 and abs(ratio_res / 38.5606 - 1) < 1e-5, (
    "khác với demo: hãy kiểm tra lại dữ liệu"
)
assert (sig(ratio_plain), sig(ratio_res)) == ("899", "38,6")

Hình dưới vẽ hai mạng có BatchNorm (thang log). Bên trái là $\lVert\partial L/\partial W_l\rVert$ của từng lớp; lượt
ngược đi từ phải (lớp 30) sang trái (lớp 1). Bên phải là độ lớn tín hiệu $\mathrm{rms}(h)$ sau từng lớp, theo lượt xuôi.
Xanh, nét liền, chấm tròn: có kết nối tắt. Cam, nét gạch, ô vuông: mạng trơn.

In [ ]:
def two_lines(ax, a, b, every=1):
    layers = np.arange(1, len(a) + 1)
    ax.plot(layers, a, **RES, ms=4, lw=1.6, markevery=every, label="có kết nối tắt: h ← h + F(h)")
    ax.plot(layers, b, **PLAIN, ms=4, lw=1.6, markevery=every, label="trơn: h ← F(h)")
    ax.set_yscale("log")
    ax.grid(axis="y", alpha=0.3, which="major")


fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
two_lines(a1, prof[("res", "bn")][0], prof[("plain", "bn")][0])
a1.set(xlabel="lớp l", ylabel="‖∂L/∂Wₗ‖", title="Gradient từng lớp (← lượt ngược)")
two_lines(a2, prof[("res", "bn")][1], prof[("plain", "bn")][1])
a2.set(xlabel="lớp l", ylabel="rms(h) sau lớp l", title="Tín hiệu sau từng lớp (lượt xuôi →)")
a1.legend(frameon=False, fontsize=9)
fig.suptitle("MLP 30 lớp, BatchNorm, lúc khởi tạo, trên 64 ảnh: cùng trọng số, chỉ khác phép cộng", fontsize=11)
plt.tight_layout()
vn_axes(fig)
plt.show()

Gradient ở lớp 1 của mạng trơn lớn gấp 899 lần ở lớp 30, đúng con số của Bài 10 và của demo. Đây là **bùng nổ**,
không phải biến mất: BatchNorm giữ tín hiệu xuôi ở cỡ không đổi (đường cam bên phải nằm ngang), nhưng chính nó làm
gradient ngược to thêm ở mỗi lớp. Yang và cộng sự (2019) chứng minh điều này cho mạng trơn có BatchNorm: không chỉnh
khởi tạo hay hàm kích hoạt nào xóa được nó; với ReLU và lô lớn, bình phương chuẩn gradient tăng $\pi/(\pi - 1)$ lần
mỗi lớp. Ô dưới so con số lý thuyết đó với số đo của ta.

Cùng trọng số, chỉ thêm phép cộng, tỉ số còn 38,6. Nó vẫn tăng dần về phía lớp đầu, vì mỗi nhánh $F$ cộng thêm phần
gradient của nó vào $\partial L/\partial h$, nhưng phần đi qua đường tắt thì không bị nhân với ma trận nào.

In [ ]:
lam = math.pi / (math.pi - 1)
print(f"lý thuyết (lô rất lớn): chuẩn gradient ×√(π/(π−1)) = ×{vn(math.sqrt(lam), 3)} mỗi lớp")
print(
    f"đo được (lô 64, 30 lớp): ×{vn(ratio_plain ** (1 / 29), 3)} mỗi lớp;  có kết nối tắt: ×{vn(ratio_res ** (1 / 29), 3)}"
)
rms_res = prof[("res", "bn")][1]
print(
    f"rms(h) có kết nối tắt: lớp 1 {vn(rms_res[0], 3)} → lớp 30 {vn(rms_res[-1], 3)}; "
    f"nếu mỗi nhánh cộng thêm phương sai 1: √({vn(rms_res[0] ** 2, 2)} + 29) = {vn(np.sqrt(rms_res[0] ** 2 + 29), 3)}"
)

Lý thuyết của Yang và cộng sự viết cho lô rất lớn và mạng rất sâu, nên chỉ nên so cỡ: ×1,21 so với ×1,26 đo được.
Còn tín hiệu của mạng có kết nối tắt thì *lớn dần* (đường xanh bên phải): mỗi nhánh nhận một đầu vào đã chuẩn hóa (phương sai 1), qua ReLU còn bình phương trung bình ½, nhân với $W$ khởi tạo
He thì thành phương sai khoảng 1, rồi cộng vào $h$. Sau 29 khối, $\mathrm{rms}(h) \approx \sqrt{\tfrac12 + 29}$, gần với
số đo. Hệ quả: ở khối thứ $l$, nhánh $F$ chỉ thay đổi $h$ cỡ $1/\sqrt{l}$, nên các khối sâu gần như là phép đồng nhất
ngay từ đầu. De & Smith (2020) cho rằng đây chính là lý do BatchNorm giúp ResNet sâu học được.

## 6 · Chỉ phép cộng thôi thì chưa đủ

### 🤔 Dự đoán trước
Bỏ BatchNorm khỏi cả hai mạng. Lúc khởi tạo, gradient của mạng trơn và mạng có kết nối tắt đều khá cân giữa lớp 1 và
lớp 30. Mạng nào sẽ học được trong 1 epoch?

In [ ]:
for a, name in (("plain", "trơn          "), ("res", "có kết nối tắt")):
    gw, rms = prof[(a, "none")]
    print(
        f"không chuẩn hóa, {name}: ‖∂L/∂W₁‖ / ‖∂L/∂W₃₀‖ = {sig(gw[0] / gw[-1]):>3}   "
        f"rms(h): lớp 1 {sig(rms[0])} → lớp 30 {sig(rms[-1])}"
    )
rms_none = prof[("res", "none")][1]
print(
    f"có kết nối tắt: rms(h) nhân ×{vn((rms_none[-1] / rms_none[0]) ** (1 / 29), 2)} mỗi lớp (√2 = {vn(math.sqrt(2), 2)})"
)
for d in mlp_runs["depths"]:  # kết quả quét: 4 giá trị η × 3 seed cho mỗi độ sâu
    rs = [r for r in mlp_runs["runs"] if (r["arch"], r["norm"], r["depth"]) == ("res", "none", d)]
    steps = [s["nanStep"] for r in rs for s in r["seeds"] if "nanStep" in s]
    eta, seeds = best_mlp("res", "none", d)
    best = f"; tốt nhất η = {vn_eta(eta)}: {mlp_text(seeds)}" if any("val" in s for s in seeds) else ""
    print(
        f"có kết nối tắt, không chuẩn hóa, {d:3d} lớp: {len(steps)}/12 lần chạy nổ (NaN), ở bước {min(steps)}{f'–{max(steps)}' if max(steps) > min(steps) else ''}{best}"
    )

Lúc khởi tạo, gradient hai mạng đều cân (tỉ số 6,7 và 5,5), nhưng mạng có kết nối tắt không chuẩn hóa **nổ** ngay
những bước đầu với mọi $\eta$ đã thử, từ 20 lớp trở lên. Lý do nằm ở đường bên phải của tín hiệu: mỗi nhánh $F$ được
khởi tạo He, nên phần nó cộng vào có phương sai cỡ phương sai của $h$; cộng vào thì phương sai **gấp đôi** mỗi khối,
$\mathrm{rms}(h)$ nhân khoảng $\sqrt2$, và không có gì kéo lại. Sau 29 khối, mất mát ban đầu đã khổng lồ, gradient cũng vậy,
nên chỉ một hai bước học đã đẩy mạng ra khỏi vùng số hữu hạn. Còn mạng trơn không chuẩn hóa thì, ở $\eta$ tốt nhất
của nó, không nổ, nhưng học kém (77,0% ở mục 3).

Vậy kết nối tắt *một mình* không đủ; phải có thứ giữ cho tổng $h + F(h)$ không phình ra. ResNet thật đặt BatchNorm
trong mỗi khối. Một cách khác là khởi tạo nhánh $F$ gần 0, để mỗi khối bắt đầu gần như là phép đồng nhất: Fixup
(Zhang và cộng sự, 2019), SkipInit (De & Smith, 2020), và GPT-2 nhân trọng số của các lớp trên nhánh phần dư với
$1/\sqrt{N}$ lúc khởi tạo. Mục “Thử thay đổi” sẽ thử cách này.

## 7 · Huấn luyện thật: 30 lớp, 1 epoch

Gradient lúc khởi tạo mới là một dấu hiệu. Ô dưới huấn luyện thật hai mạng 30 lớp có BatchNorm như demo: SGD, lô 64,
1 epoch qua 55.000 ảnh với thứ tự `default_rng(1)`, seed 1. Mỗi mạng dùng $\eta$ tốt nhất của nó trong lưới
0,01 · 0,03 · 0,1 · 0,3, chọn theo tập kiểm định trong kết quả quét. Mỗi mạng mất vài giây.

In [ ]:
def accuracy(P, X, y):
    return float(
        np.mean([(forward(P, X[i : i + 2500], "eval")[2].argmax(1) == y[i : i + 2500]) for i in range(0, len(X), 2500)])
    )


def train(arch, norm, depth, eta, seed=1, branch_scale=1.0):
    """1 epoch SGD; branch_scale nhân trọng số các nhánh F lúc khởi tạo (chỉ dùng ở mục Thử thay đổi)."""
    np.seterr(all="ignore")  # mạng nổ sẽ tràn số; ta phát hiện bằng mất mát bên dưới
    P = make_net(arch, norm, depth, seed)
    P["W"][1:depth] = [w * branch_scale for w in P["W"][1:depth]]
    order, losses = np.random.default_rng(seed).permutation(N_TRAIN), []
    for step, i in enumerate(range(0, N_TRAIN, BATCH), 1):
        idx = order[i : i + BATCH]
        cache, h, z, _ = forward(P, Xtr[idx], "train")
        zmax = z.max(1)  # log-sum-exp ổn định, như các script của demo
        losses.append((zmax + np.log(np.exp(z - zmax[:, None]).sum(1)) - z[np.arange(len(idx)), ytr[idx]]).mean())
        if not np.isfinite(losses[-1]):
            return None, step, losses
        g = backward(P, cache, h, z, ytr[idx])
        for k in ("W", "b", "g", "be"):
            for l, gk in enumerate(g[k]):
                if gk is not None:
                    P[k][l] -= eta * gk
    return P, step, losses

In [ ]:
nets, curves = {}, {}
for arch in ("plain", "res"):
    eta, seeds = best_mlp(arch, "bn", 30)
    t0 = time.perf_counter()
    nets[arch], _, curves[arch] = train(arch, "bn", 30, eta)
    acc = accuracy(nets[arch], Xv, yv)
    print(
        f"{'trơn          ' if arch == 'plain' else 'có kết nối tắt'} (η = {vn_eta(eta)}): đúng {pct(acc)} trên tập kiểm định "
        f"({vn(time.perf_counter() - t0, 1)} giây); kết quả quét, seed 1: {pct(seeds[0]['val'])}"
    )
    nets[arch]["acc"] = acc
assert abs(nets["res"]["acc"] - 0.9608) < 0.002, "khác với demo: hãy kiểm tra lại dữ liệu"
assert nets["plain"]["acc"] < nets["res"]["acc"] - 0.1

Hình dưới vẽ mất mát huấn luyện của 1 epoch đó (trung bình trượt 20 bước, thang log): xanh nét liền là mạng có kết nối
tắt, cam nét gạch là mạng trơn.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
for arch, st, label in (("res", RES, "có kết nối tắt"), ("plain", PLAIN, "trơn")):
    smooth = np.convolve(curves[arch], np.ones(20) / 20, mode="valid")
    ax.plot(np.arange(20, len(curves[arch]) + 1), smooth, color=st["color"], ls=st["ls"], lw=1.6, label=label)
ax.set_yscale("log")
ax.set(xlabel="bước (1 epoch = 860 bước)", ylabel="mất mát huấn luyện", title="MLP 30 lớp có BatchNorm, seed 1")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.3, which="both")
vn_axes(fig)
plt.show()

Mạng có kết nối tắt học được ngay: con số trùng với kết quả quét mà demo lưu. Mạng trơn tụt lại rất xa, dù hai mạng có
cùng trọng số khởi tạo, cùng dữ liệu, cùng thứ tự lô.

Với mạng trơn, số của bạn có thể lệch khỏi số trong kết quả quét vài điểm phần trăm, và notebook không `assert` nó.
Lý do là chính điều bài này nói: gradient của mạng trơn bùng nổ, nên một sai khác làm tròn ở chữ số thứ 16 (Colab
chạy trên x86, kết quả quét chạy trên ARM) cũng bị khuếch đại thành một đường học khác. Mục “Thử thay đổi” đo điều này.

## 8 · Kết quả quét: từ 10 đến 100 lớp, và CNN đến 56 lớp

Quét đầy đủ (5 độ sâu × 4 giá trị $\eta$ × 3 seed cho mỗi mạng) mất vài phút, nên ta đọc kết quả mà demo đã lưu.
Mỗi độ sâu lấy $\eta$ có độ chính xác kiểm định trung vị cao nhất.

In [ ]:
print("MLP, BatchNorm, 1 epoch — đúng trên tập kiểm định, trung vị [min–max] của 3 seed:")
mlp_best = {}
for d in mlp_runs["depths"]:
    for arch in ("plain", "res"):
        mlp_best[(arch, d)] = best_mlp(arch, "bn", d)
    (ep, sp), (er, sr) = mlp_best[("plain", d)], mlp_best[("res", d)]
    print(f"  {d:3d} lớp: trơn {mlp_text(sp)} (η = {vn_eta(ep)}) · có kết nối tắt {mlp_text(sr)} (η = {vn_eta(er)})")

Hình dưới vẽ bảng đó: mỗi dấu là một seed, đường nối trung vị. Đường gạch xám ở 10% là mức đoán mò.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
D = mlp_runs["depths"]
for arch, st, label in (("res", RES, "có kết nối tắt"), ("plain", PLAIN, "trơn")):
    v = [[s["val"] for s in mlp_best[(arch, d)][1]] for d in D]
    for d, vs in zip(D, v):
        ax.scatter([d] * 3, 100 * np.array(vs), color=st["color"], marker=st["marker"], s=26, zorder=3)
    ax.plot(D, [100 * np.median(vs) for vs in v], **st, ms=5, mfc="white", lw=1.6, label=label)
ax.axhline(10, color=GREY, ls="--", lw=1)
ax.set_xscale("log")
ax.set(xlabel="số lớp ẩn (thang log)", ylabel="đúng trên tập kiểm định (%)", ylim=(0, 100))
ax.set_title("MLP có BatchNorm, 1 epoch, η tốt nhất cho mỗi độ sâu")
ax.legend(frameon=False, loc="center left")
ax.grid(axis="y", alpha=0.3)
vn_axes(fig)
ax.xaxis.set_major_locator(FixedLocator(D))
plt.show()

Có kết nối tắt, độ chính xác gần như không đổi từ 10 đến 100 lớp. Không có, mạng trơn sụp từ 30 lớp và ở 100 lớp chỉ
còn quanh mức đoán mò. Lưu ý: kết nối tắt không làm mạng 100 lớp *tốt hơn* mạng 10 lớp ở đây (96,4% so với 96,3%,
trong dao động giữa các seed; MNIST và 1 epoch không cần nhiều lớp đến thế); nó làm cho việc thêm lớp **không còn hại**.

Giờ quay lại đúng các mạng CNN của Bài 12 (mục 3). `resnet.py` biến mỗi cặp lớp tích chập 3×3 thành một khối phần dư
cơ bản của He và cộng sự (2015): conv → BN → ReLU → conv → BN, cộng đường tắt, rồi ReLU. Ở khối thu nhỏ ảnh, đường tắt
lấy mỗi điểm ảnh thứ hai và thêm các kênh 0 (phương án A). Hai mạng có cùng số lớp, cùng trọng số khởi tạo, cùng
lưới $\eta$ và seed, và **cùng số tham số**, vì đường tắt không có tham số nào.

In [ ]:
R_cnn = best_cnn(cnn_res)
for d in depths:
    p, r = P_cnn[d], R_cnn[d]
    print(
        f"{d:2d} lớp ({vn(p['params'][0], 0):>7} tham số): lỗi huấn luyện trơn {seed_text(p['train'])} · "
        f"có kết nối tắt {seed_text(r['train'])}"
    )
for d in (20, 56):
    q = min((q for q in cnn_res["long"] if q["depth"] == d), key=lambda q: q["val"])
    print(
        f"10 epoch, seed 0, {d} lớp có kết nối tắt (η = {vn_eta(q['eta'])}): lỗi huấn luyện {pct(q['train'])}, "
        f"lỗi kiểm tra {pct(q['test'])}"
    )
g = cnn_res["initGrads"]
print(
    f"gradient lúc khởi tạo, lớp tích chập đầu ÷ cuối, 56 lớp: trơn {sig(g['plain']['56'][0] / g['plain']['56'][-1])} · "
    f"có kết nối tắt {sig(g['res']['56'][0] / g['res']['56'][-1])}"
)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.4))
for grid, st, label in ((R_cnn, RES, "có kết nối tắt"), (P_cnn, PLAIN, "trơn (Bài 12)")):
    for d in depths:
        ax.scatter([d] * 3, 100 * np.array(grid[d]["train"]), color=st["color"], marker=st["marker"], s=26, zorder=3)
    ax.plot(depths, [100 * np.median(grid[d]["train"]) for d in depths], **st, ms=5, mfc="white", lw=1.6, label=label)
ax.set(xticks=depths, xlabel="số lớp", ylabel="lỗi huấn luyện (%)", ylim=(0, None))
ax.set_title("CNN của Bài 12 trên MNIST, 3 epoch, thêm kết nối tắt")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.3)
vn_axes(fig)
plt.show()

Với kết nối tắt, đường cong lỗi huấn luyện không còn đi lên theo độ sâu: ở 56 lớp, khoảng 3 seed của mạng có kết nối
tắt nằm hẳn dưới khoảng của mạng trơn. Học 10 epoch (một seed) thì mạng 56 lớp có lỗi huấn luyện *thấp hơn* mạng
20 lớp, giống điều He và cộng sự thấy trên ImageNet: ResNet-34 có lỗi huấn luyện thấp hơn hẳn ResNet-18, và lỗi top‑1
25,03% so với 27,88% (Bảng 2). Ở 8 lớp hai mạng ngang nhau, trong dao động giữa các seed: mạng nông thì chưa có gì để
sửa. Các con số nhỏ hơn bài báo nhiều vì mạng nhỏ và học ngắn; hình dạng của đường cong mới là điều cần so. Gradient
lúc khởi tạo của CNN trơn 56 lớp cũng lệch gần 1.000 lần giữa lớp đầu và lớp cuối (946), như MLP trơn ở mục 5; có kết
nối tắt thì chỉ 2,5 lần.

## 9 · Xóa một khối: mạng phần dư như một tập hợp nhiều đường đi

Veit và cộng sự (2016) làm một thí nghiệm đơn giản: lấy một mạng đã học xong, **xóa** một khối (không học lại), rồi
đo lỗi. Với khối phần dư, “xóa” nghĩa là chỉ giữ đường tắt: $h \leftarrow h$. Với mạng trơn, xóa một lớp
$64 \to 64$ nghĩa là nối thẳng đầu vào của nó tới lớp kế tiếp.

### 🤔 Dự đoán trước
Hai mạng 30 lớp vừa học ở mục 7. Xóa *một* lớp ở giữa (bất kỳ lớp nào trong số 29 lớp 64 → 64). Độ chính xác của
mỗi mạng còn khoảng bao nhiêu?

In [ ]:
def drop_layer(P, l):
    """Bản sao của mạng đã học, bỏ lớp ẩn l (1 … L−1): khối phần dư chỉ còn đường tắt; lớp trơn bị nối tắt qua."""
    keep_W = [i for i in range(len(P["W"])) if i != l]
    keep_n = [i for i in range(len(P["g"])) if i != l]
    Q = {**P, "depth": P["depth"] - 1, "W": [P["W"][i] for i in keep_W], "b": [P["b"][i] for i in keep_W]}
    Q.update({k: [P[k][i] for i in keep_n] for k in ("g", "be", "rm", "rv")})
    return Q


Xl, yl = Xv[:2000], yv[:2000]  # 2.000 ảnh kiểm định cho nhanh, như thí nghiệm CNN bên dưới dùng 2.000 ảnh
lesion = {arch: np.array([accuracy(drop_layer(nets[arch], l), Xl, yl) for l in range(1, 30)]) for arch in nets}
full_l = {arch: accuracy(nets[arch], Xl, yl) for arch in nets}
for arch, name in (("res", "có kết nối tắt"), ("plain", "trơn          ")):
    v = lesion[arch]
    print(
        f"{name}: đầy đủ {pct(full_l[arch], 1)}; xóa 1 lớp → trung vị {pct(np.median(v), 1)} "
        f"[{pct(v.min(), 1)}–{pct(v.max(), 1)}] trên 29 lớp"
    )
print(f"mạng có kết nối tắt: xóa lớp {int(np.argmin(lesion['res'])) + 2} là tệ nhất")


def branch_share(P, X):
    """‖F(h)‖ / ‖h‖ của từng khối trong mạng có kết nối tắt đã học (chế độ suy luận)."""
    y, _ = norm_fwd(P, 0, X @ P["W"][0] + P["b"][0], "eval")
    h, out = relu(y), []
    for l in range(1, P["depth"]):
        y, _ = norm_fwd(P, l, h, "eval")
        f = relu(y) @ P["W"][l] + P["b"][l]
        out.append(np.linalg.norm(f) / np.linalg.norm(h))
        h = h + f
    return np.array(out)


share = branch_share(nets["res"], Xv[:2000])
print(
    f"‖F(h)‖ / ‖h‖: khối ở lớp 2 {vn(share[0], 2)}; các khối ở lớp 3–30 {vn(share[1:].min(), 2)}–{vn(share[1:].max(), 2)}"
)

Hình dưới vẽ độ chính xác kiểm định sau khi xóa từng lớp (trục ngang: lớp bị xóa), xanh chấm tròn là mạng có kết nối
tắt, cam ô vuông là mạng trơn; hai đường ngang mảnh là độ chính xác của mạng đầy đủ.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
for arch, st, label in (("res", RES, "có kết nối tắt: xóa khối"), ("plain", PLAIN, "trơn: xóa lớp")):
    ax.plot(np.arange(2, 31), 100 * lesion[arch], **st, ms=4.5, lw=1.2, label=label)
    ax.axhline(100 * full_l[arch], color=st["color"], ls=":", lw=1)
ax.set(xlabel="lớp bị xóa", ylabel="đúng trên 2.000 ảnh kiểm định (%)", ylim=(0, 100), xlim=(1, 31))
ax.set_title("MLP 30 lớp đã học: xóa một lớp, không học lại")
ax.legend(frameon=False, loc="center right")
ax.grid(axis="y", alpha=0.3)
vn_axes(fig)
plt.show()

Mạng trơn sụp về gần mức đoán mò dù chỉ mất một trong 30 lớp: mỗi lớp nhận đúng đầu ra của lớp trước nó, nên bỏ một
mắt xích là cả chuỗi phía sau nhận dữ liệu lạ. Mạng có kết nối tắt gần như không đổi, trừ khối đầu tiên (lớp 2): ở
đó nhánh $F$ còn lớn hơn cả $h$ mà nó được cộng vào (dòng cuối của ô xóa lớp, ‖F(h)‖ / ‖h‖ = 1,96), còn ở các khối sau
nó chỉ là một phần hiệu chỉnh nhỏ.

Lý do: khai triển $h_3 = h_2 + F_3(h_2)$ với $h_2 = h_1 + F_2(h_1)$ cho thấy mạng $n$ khối là tổng của $2^n$ **đường đi**,
mỗi đường đi qua hoặc bỏ qua từng khối. Xóa một khối chỉ bỏ đi một nửa số đường; các đường còn lại vẫn cho ra kết quả
hợp lý. Veit và cộng sự gọi đó là hành vi giống một **tập hợp** (ensemble) các mạng nông hơn, và đo được rằng hầu hết
gradient khi học đi qua các đường ngắn: ở ResNet 110 lớp, các đường dài 10–34 lớp.

Thí nghiệm của Veit dùng mạng tích chập. `resnet.py` làm đúng thí nghiệm đó trên hai CNN 56 lớp ở mục 8 ($\eta = 0{,}1$,
3 seed), đo lỗi trên 2.000 ảnh kiểm tra: xóa từng khối một, rồi xóa $k$ khối ngẫu nhiên cùng lúc (20 lần rút cho mỗi $k$).
Mạng trơn được xóa từng lớp Conv → BN → ReLU giữ nguyên kích thước (52 lớp như vậy).

In [ ]:
def med(a):
    """Trung vị như demo: với số phần tử chẵn, lấy phần tử giữa phía trên."""
    return sorted(a)[len(a) // 2]


LS = cnn_res["lesion"]
print(
    f"{LS['depth']} lớp, η = {vn_eta(LS['eta'])}; lỗi đo trên {vn(LS['nTest'], 0)} ảnh kiểm tra, {LS['draws']} lần rút cho mỗi k"
)
full = {a: [r["full"] for r in LS[a]] for a in ("res", "plain")}
single = {a: [med([r["single"][u] for r in LS[a]]) for u in range(LS[a][0]["units"])] for a in ("res", "plain")}
for a, name, unit in (("res", "có kết nối tắt", "khối"), ("plain", "trơn", "lớp")):
    print(
        f"CNN {name}: {LS[a][0]['units']} {unit} xóa được; lỗi kiểm tra đầy đủ {seed_text(full[a], 1)}, xóa 1 {unit} → "
        f"trung vị {pct(med(single[a]), 1)} (theo từng {unit}: {pct(min(single[a]), 1)}–{pct(max(single[a]), 1)})"
    )
heavy = [(u + 1, v) for u, v in enumerate(single["res"]) if v > 3 * med(full["res"])]
print("khối làm lỗi tăng hơn 3 lần khi bị xóa:", ", ".join(f"khối {u} ({pct(v, 1)})" for u, v in heavy))
per_stage = LS["res"][0]["units"] // 3
print("khối thu nhỏ ảnh:", ", ".join(f"khối {u + 1} ({pct(single['res'][u], 1)})" for u in (per_stage, 2 * per_stage)))


def pooled(a):
    """Cho mỗi k: mọi lần rút của cả 3 seed gộp lại (k = 0 là mạng đầy đủ)."""
    return [full[a]] + [list(itertools.chain(*(r["random"][i] for r in LS[a]))) for i in range(len(LS["ks"]))]


series = {a: [med(v) for v in pooled(a)] for a in full}
print(
    "xóa k khối ngẫu nhiên, lỗi trung vị:",
    " · ".join(f"k = {k}: {pct(v, 1)}" for k, v in zip([0, *LS["ks"]], series["res"])),
)

Hình dưới vẽ lỗi kiểm tra của hai CNN 56 lớp theo số khối (hoặc lớp) bị xóa ngẫu nhiên cùng lúc: đường là trung vị
của 60 lần rút (3 seed × 20), vùng tô chứa nửa giữa của các lần rút (từ tứ phân vị dưới đến tứ phân vị trên). Xanh
chấm tròn: có kết nối tắt; cam ô vuông: mạng trơn.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
ks = [0, *LS["ks"]]
for a, st, label in (("res", RES, "có kết nối tắt: xóa k khối"), ("plain", PLAIN, "trơn: xóa k lớp")):
    draws = [sorted(v) for v in pooled(a)]
    q = np.array([[d[round(f * (len(d) - 1))] for f in (0.25, 0.75)] for d in draws])
    ax.fill_between(ks, 100 * q[:, 0], 100 * q[:, 1], color=st["color"], alpha=0.15, lw=0)
    ax.plot(ks, 100 * np.array(series[a]), **st, ms=4.5, lw=1.6, label=label)
ax.axhline(90, color=GREY, ls="--", lw=1)
ax.text(0.1, 92, "đoán mò", color=GREY, fontsize=9, ha="left")
ax.set(xticks=ks, xlabel="số khối / lớp bị xóa", ylabel="lỗi kiểm tra (%)", ylim=(0, 100))
ax.set_title("CNN 56 lớp đã học, 2.000 ảnh kiểm tra: xóa ngẫu nhiên, không học lại")
ax.legend(frameon=False, loc="center right")
ax.grid(axis="y", alpha=0.3)
vn_axes(fig)
plt.show()

CNN cho cùng bức tranh: xóa một lớp của mạng trơn là hỏng hẳn, còn xóa một khối của mạng phần dư thường chỉ làm lỗi
tăng nhẹ. Lỗi tăng **dần** khi xóa nhiều khối, không sụp một lần, đúng như Veit và cộng sự mô tả. Ba khối đầu của
chặng 1 là ngoại lệ, giống khối đầu của MLP ở trên. Veit thấy xóa một khối thu nhỏ ảnh có tác
động vừa phải, rõ hơn các khối khác; ở đây hai khối đó (khối 10 và 19) cho 3,4% và 4,9%, nhỉnh hơn trung vị một chút.
Xóa 10 khối thì lỗi lên 84,4%, cao hơn nhiều so với bài báo, vì ở đây 10 khối là hơn một phần ba mạng (bài báo xóa
10 trong 54 khối).

Veit và cộng sự rút ra một kết luận đáng nhớ: ResNet không giữ cho gradient đi nguyên vẹn qua *toàn bộ* độ sâu; nó
thêm các đường ngắn mang gradient đi xuyên mạng, và việc học diễn ra chủ yếu trên những đường ấy.

## 10 · Thử thay đổi

Các ô dưới chạy lại cùng các hàm với tham số khác. Bạn có thể sửa giá trị rồi chạy lại; mỗi lần huấn luyện mất vài giây.

### Kết nối tắt không cần BatchNorm, nếu nhánh bắt đầu nhỏ
Mục 6 thấy mạng có kết nối tắt mà không chuẩn hóa nổ ngay ở bước 3. Ý của Fixup và SkipInit: nhân trọng số mỗi nhánh
$F$ với một số nhỏ lúc khởi tạo, để $h + F(h)$ bắt đầu gần $h$. Ô dưới thử ba hệ số trên mạng 30 lớp không chuẩn hóa,
$\eta = 0{,}03$ (thử cả $1/\sqrt{29}$, cỡ hệ số GPT-2 dùng).

In [ ]:
pred = {}
for scale in (1.0, 1 / np.sqrt(29), 0.0):
    P, step, losses = train("res", "none", 30, 0.03, branch_scale=scale)
    if P is None:
        print(f"nhánh F khởi tạo ×{vn(scale, 3)}: nổ (NaN) ở bước {step}")
        assert scale == 1.0 and step <= 4, "khác với demo: mạng không chuẩn hóa phải nổ trong vài bước đầu"
        continue
    pred[scale] = forward(P, Xv, "eval")[2].argmax(1)
    print(f"nhánh F khởi tạo ×{vn(scale, 3)}: đúng {pct((pred[scale] == yv).mean())} trên tập kiểm định")
a, b = pred.values()
print(f"hai mạng học được đưa ra cùng câu trả lời cho {pct((a == b).mean())} số ảnh kiểm định")

Hai độ chính xác bằng nhau chỉ là trùng hợp: hai mạng trả lời khác nhau ở vài phần trăm số ảnh. Với nhánh nhân 0
lúc khởi tạo, mạng ban đầu chỉ là lớp 1 cộng lớp ra; các khối lớn dần trong lúc học. Cả hai cách đều học được 30 lớp
có kết nối tắt mà không cần BatchNorm. Điều kiện thật sự là tổng $h + F(h)$ không được phình ra theo độ sâu; BatchNorm
là một cách, khởi tạo nhánh nhỏ là một cách khác.

### Mạng trơn nhạy với một sai khác cực nhỏ
Mục 7 nói số của mạng trơn có thể lệch trên máy khác. Ô dưới đo điều đó: chạy hai mạng 100 bước, một lần với trọng số
gốc, một lần với $W_1$ nhân $(1 + 10^{-12}\varepsilon)$, rồi đo khoảng cách tương đối giữa hai bộ trọng số.

In [ ]:
def drift(arch, eta, steps=100, rel=1e-12):
    runs = []
    for r in (0.0, rel):
        P = make_net(arch, "bn", 30)
        P["W"][0] = P["W"][0] * (1 + r * np.random.default_rng(99).standard_normal(P["W"][0].shape))
        order = np.random.default_rng(1).permutation(N_TRAIN)
        for i in range(0, steps * BATCH, BATCH):
            idx = order[i : i + BATCH]
            cache, h, z, _ = forward(P, Xtr[idx], "train")
            g = backward(P, cache, h, z, ytr[idx])
            for k in ("W", "b", "g", "be"):
                for l, gk in enumerate(g[k]):
                    if gk is not None:
                        P[k][l] -= eta * gk
        runs.append(np.concatenate([w.ravel() for w in P["W"]]))
    return np.linalg.norm(runs[1] - runs[0]) / np.linalg.norm(runs[0])


for arch, name in (("plain", "trơn          "), ("res", "có kết nối tắt")):
    print(
        f"{name}: sau 100 bước, hai bộ trọng số cách nhau {sig(drift(arch, best_mlp(arch, 'bn', 30)[0]))} (tương đối)"
    )

Sau 100 bước, sai khác $10^{-12}$ ở mạng trơn đã lớn thành cỡ một phần năm độ lớn của trọng số: hai lần chạy là hai
mạng khác hẳn nhau. Ở mạng có kết nối tắt, sai khác giữ nguyên cỡ ban đầu. Đây là một mặt khác của cùng một vấn đề:
mạng trơn sâu là một hệ khuếch đại nhiễu, đến mức kết quả của nó phụ thuộc vào cách máy làm tròn số. Balduzzi và cộng
sự (2017) mô tả hiện tượng gần với điều này ở gradient: trong mạng trơn sâu, gradient của các đầu vào gần nhau trở nên
gần như không tương quan (“gradient vỡ vụn”), còn kết nối tắt giữ chúng tương quan.

### Độ sâu khác
Đổi `depth` để xem tỉ số gradient lúc khởi tạo thay đổi thế nào theo độ sâu (demo cho chọn 10, 20, 30, 50, 100 lớp).

In [ ]:
for depth in (10, 20, 50, 100):
    rp, rr = (grad_profile(a, "bn", depth)[0] for a in ("plain", "res"))
    print(
        f"{depth:3d} lớp, BatchNorm: ‖∂L/∂W₁‖ / ‖∂L/∂W_L‖ trơn {sig(rp[0] / rp[-1]):>9} · có kết nối tắt {sig(rr[0] / rr[-1])}"
    )

Ở mạng trơn tỉ số tăng theo hàm mũ của độ sâu (mỗi lớp nhân một hệ số gần như cố định); có kết nối tắt thì nó chỉ tăng
chậm. Thử thêm: đổi `seed` trong `train(...)` của mục 7, hoặc đổi `eta`, rồi so với khoảng 3 seed ở mục 8.

## 11 · Giới hạn và bước tiếp theo

- **Kết nối tắt sửa việc tối ưu, không sửa quá khớp.** Trên CIFAR-10, He và cộng sự huấn luyện được mạng 1.202 lớp
  (19,4 triệu tham số), nhưng lỗi kiểm tra 7,93% của nó cao hơn mạng 110 lớp (6,43%); các tác giả cho là do quá khớp.
- **Không phải lời giải thích duy nhất.** Vì sao kết nối tắt giúp vẫn còn được tranh luận: đường cho gradient đi thẳng,
  khối dễ học phép đồng nhất, tập hợp các đường ngắn (Veit), gradient không vỡ vụn (Balduzzi), mặt mất mát dễ tối ưu
  hơn (Li và cộng sự, 2018). Các cách nhìn này bổ sung cho nhau.
- **Kiến trúc thật.** ResNet-18/34 dùng khối cơ bản của mục 8 (ReLU sau phép cộng); ResNet-50/101/152 dùng khối thắt cổ
  chai 1×1 → 3×3 → 1×1 để giảm chi phí. Highway Networks (Srivastava và cộng sự, 2015) đã có đường tắt trước đó, nhưng
  có cổng học được.

Ô cuối đếm số tham số của 5 mạng ResNet trên ImageNet từ cấu trúc của chúng (ảnh màu, 1.000 lớp), để thấy “152 lớp”
không có nghĩa là lớn gấp 8 lần “18 lớp”.

In [ ]:
def resnet_params(blocks, bottleneck):
    """Tham số của ResNet cho ImageNet (He và cộng sự 2015, Bảng 1): conv 7×7, 4 chặng, dense 1.000 lớp.
    Tích chập không có độ lệch (BN đi sau); BN có γ, β; đường tắt đổi kích thước dùng tích chập 1×1 (phương án B)."""
    conv = lambda cin, cout, k: cin * cout * k * k + 2 * cout  # trọng số + γ, β của BN đi sau
    p, cin, mult = conv(3, 64, 7), 64, 4 if bottleneck else 1
    for stage, (n, w) in enumerate(zip(blocks, (64, 128, 256, 512))):
        for i in range(n):
            cout = w * mult
            if bottleneck:  # khối thắt cổ chai 1×1 → 3×3 → 1×1
                p += conv(cin, w, 1) + conv(w, w, 3) + conv(w, cout, 1)
            else:  # khối cơ bản 3×3 → 3×3
                p += conv(cin, w, 3) + conv(w, w, 3)
            if cin != cout or (stage > 0 and i == 0):
                p += conv(cin, cout, 1)
            cin = cout
    return p + cin * 1000 + 1000


CONFIGS = {18: ([2, 2, 2, 2], False), 34: ([3, 4, 6, 3], False), 50: ([3, 4, 6, 3], True),
           101: ([3, 4, 23, 3], True), 152: ([3, 8, 36, 3], True)}  # fmt: skip
counts = {d: resnet_params(*c) for d, c in CONFIGS.items()}
for d, n in counts.items():
    print(f"ResNet-{d:<3}: {vn(n, 0):>10} tham số ({vn(n / counts[18], 1)} lần ResNet-18)")
# số tham số mà torchvision công bố cho các mô hình của nó
assert list(counts.values()) == [11689512, 21797672, 25557032, 44549160, 60192808]


def cifar_params(depth):
    """ResNet cho CIFAR-10 (mục 4.2 của bài báo): 3 chặng 16/32/64 kênh, khối cơ bản, đường tắt phương án A."""
    conv = lambda cin, cout: cin * cout * 9 + 2 * cout
    p, cin = conv(3, 16), 16
    for w in (16, 32, 64):
        for _ in range((depth - 2) // 6):
            p, cin = p + conv(cin, w) + conv(w, w), w
    return p + 64 * 10 + 10


for d in (110, 1202):
    print(f"ResNet {vn(d, 0)} lớp cho CIFAR-10: {vn(cifar_params(d) / 1e6, 1)} triệu tham số")

ResNet-152 sâu gấp hơn 8 lần ResNet-18 nhưng chỉ nhiều tham số hơn khoảng 5 lần, nhờ khối thắt cổ chai. Đó cũng là
lý do ResNet-50 chỉ hơn ResNet-34 một chút về tham số dù sâu hơn 16 lớp.

Kết nối tắt không dừng ở thị giác máy tính. Trong Transformer (Bài 17), mỗi khối attention và mỗi khối MLP đều viết
$h \leftarrow h + F(\mathrm{Norm}(h))$, đúng dạng tiền kích hoạt của mục 5: một “dòng phần dư” chạy từ embedding tới
lớp ra, và mỗi khối chỉ cộng thêm phần hiệu chỉnh của nó. Bài 14 bắt đầu chặng đó, với đầu vào của Transformer:
token và embedding.

## Tóm tắt
- **Vấn đề suy thoái:** CNN trơn 56 lớp có lỗi *huấn luyện* 9,41% so với 0,99% của mạng 20 lớp. Đó là thất bại của
  tối ưu, không phải quá khớp, và BatchNorm không cứu được.
- **Khối phần dư** $y = x + F(x)$: với $F = 0$ nó là phép đồng nhất, và Jacobi của nó là $I + \partial F/\partial x$.
- **Lúc khởi tạo:** mạng MLP trơn 30 lớp có BatchNorm có gradient lớp 1 gấp 899 lần lớp 30 (bùng nổ, không phải biến
  mất). Cùng trọng số, thêm phép cộng: 38,6 lần.
- **Chỉ phép cộng thôi chưa đủ:** không chuẩn hóa, phương sai của $h$ gấp đôi mỗi khối và mạng nổ ở bước 3. BatchNorm,
  hoặc khởi tạo nhánh nhỏ (Fixup, SkipInit), giữ cho tổng không phình ra.
- **Huấn luyện thật:** MLP 30 lớp, 1 epoch: 96,08% có kết nối tắt so với 65,96% trơn (seed 1). Có kết nối tắt, độ chính
  xác gần như không đổi từ 10 đến 100 lớp. CNN 56 lớp có kết nối tắt có lỗi huấn luyện 0,99% [0,78%–1,34%], ngang
  mạng trơn 20 lớp (0,99% [0,87%–1,10%]) chứ không còn 9,41%.
- **Xóa một khối** của mạng phần dư đã học thường chỉ làm lỗi tăng nhẹ; xóa một lớp của mạng trơn làm nó sụp về gần
  mức đoán mò. Mạng phần dư giống một tập hợp nhiều đường ngắn (Veit và cộng sự, 2016).

## Câu hỏi ôn tập
1. Mạng 56 lớp có lỗi huấn luyện cao hơn mạng 20 lớp. Vì sao đây không phải quá khớp?
2. Viết đạo hàm $\partial y/\partial x$ của $y = x + \mathrm{ReLU}(x)W + b$. Phần nào bảo đảm gradient không bị
   triệt tiêu khi đi qua khối?
3. Mạng trơn 30 lớp có BatchNorm: gradient ở lớp 1 lớn hay nhỏ hơn ở lớp 30? Điều đó nói gì về câu “ResNet sửa gradient
   biến mất”?
4. Vì sao mạng có kết nối tắt mà không chuẩn hóa lại nổ, dù gradient lúc khởi tạo của nó cân hơn cả mạng trơn?
   Kể hai cách sửa.
5. Xóa một khối của ResNet đã học, lỗi chỉ tăng nhẹ; xóa một lớp của mạng trơn, mạng sụp. Giải thích bằng số đường đi.
6. Đường tắt có bao nhiêu tham số? Vì sao điều này quan trọng khi so mạng trơn với mạng có kết nối tắt ở mục 8?

<details><summary>Gợi ý đáp án</summary>

1. Quá khớp là lỗi huấn luyện thấp nhưng lỗi kiểm tra cao. Ở đây chính lỗi huấn luyện cao, dù mạng 56 lớp có thể làm
   ít nhất tốt bằng mạng 20 lớp (cho các lớp thêm vào làm phép đồng nhất): tối ưu thất bại.
2. $\partial y_j/\partial x_i = \delta_{ij} + \mathbb 1[x_i > 0]\,W_{ij}$, tức $I + \partial F/\partial x$. Phần $I$ của
   đường tắt chuyển $\partial L/\partial y$ về nguyên vẹn, không nhân với ma trận nào.
3. Lớn hơn, 899 lần: BatchNorm làm gradient của mạng trơn bùng nổ theo độ sâu (Yang và cộng sự, 2019). He và cộng sự
   cũng kiểm tra rằng gradient của mạng trơn có BatchNorm không biến mất. Vấn đề của mạng trơn sâu không chỉ là
   gradient nhỏ.
4. Mỗi nhánh khởi tạo He cộng vào $h$ một phần có phương sai cỡ phương sai của $h$, nên phương sai gấp đôi mỗi khối
   ($\mathrm{rms}(h)$ ×1,42 mỗi lớp); mất mát và gradient ban đầu khổng lồ, nên vài bước học đã làm tràn số.
   Sửa: BatchNorm trong mỗi khối, hoặc khởi tạo nhánh $F$ gần 0 (Fixup, SkipInit, hệ số $1/\sqrt{N}$ của GPT-2).
5. Mạng $n$ khối là tổng của $2^n$ đường đi; xóa một khối còn lại $2^{n-1}$ đường, vẫn cho kết quả hợp lý. Mạng trơn chỉ
   có một đường, bỏ một mắt xích là đứt.
6. Không có tham số nào (phương án A). Hai mạng có đúng cùng số tham số và cùng trọng số khởi tạo, nên khác biệt chỉ có
   thể do phép cộng.
</details>

## Tìm hiểu thêm
- K. He, X. Zhang, S. Ren, J. Sun (2015), [Deep Residual Learning for Image Recognition](https://arxiv.org/abs/1512.03385):
  vấn đề suy thoái, khối phần dư, phương án A/B, CIFAR-10 đến 1.202 lớp.
- K. He và cộng sự (2016), [Identity Mappings in Deep Residual Networks](https://arxiv.org/abs/1603.05027): khối tiền
  kích hoạt, đường tắt là phép đồng nhất thuần túy.
- A. Veit, M. Wilber, S. Belongie (2016), [Residual Networks Behave Like Ensembles of Relatively Shallow Networks](https://arxiv.org/abs/1605.06431):
  thí nghiệm xóa khối, các đường ngắn mang gradient.
- G. Yang và cộng sự (2019), [A Mean Field Theory of Batch Normalization](https://arxiv.org/abs/1902.08129): BatchNorm
  làm gradient của mạng trơn sâu bùng nổ.
- S. De, S. Smith (2020), [Batch Normalization Biases Residual Blocks Towards the Identity Function in Deep Networks](https://arxiv.org/abs/2002.10444)
  và H. Zhang, Y. Dauphin, T. Ma (2019), [Fixup Initialization](https://arxiv.org/abs/1901.09321): mạng phần dư không cần
  chuẩn hóa.
- D. Balduzzi và cộng sự (2017), [The Shattered Gradients Problem](https://arxiv.org/abs/1702.08591);
  H. Li và cộng sự (2018), [Visualizing the Loss Landscape of Neural Nets](https://arxiv.org/abs/1712.09913).
- R. Srivastava, K. Greff, J. Schmidhuber (2015), [Highway Networks](https://arxiv.org/abs/1505.00387): đường tắt có cổng.
- d2l.ai, [Residual Networks (ResNet)](https://d2l.ai/chapter_convolutional-modern/resnet.html): cùng ý tưởng qua lớp
  hàm lồng nhau, có code ResNet-18.

**Bài tiếp theo:** [Bài 14 · Token & embedding](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/token-embedding/): biến
văn bản thành dãy vectơ, đầu vào của Transformer.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")